# Day 28 — Full Working Prediction Center

Run this notebook from your IPL repository in VS Code/Jupyter using the `ipl-player-performance` environment.

It backs up relevant files, detects existing routes, creates a Prediction Center page at `/api/predictions/`, adds a link from the current dashboard, runs Django checks/smoke tests, and writes `docs/day28_working_report.md`.

It does not overwrite model code or invent feature inputs. Use the exact JSON payload expected by your existing prediction view.


In [ ]:
from pathlib import Path
import re, json, shutil, datetime, subprocess, sys

CWD = Path.cwd().resolve()
ROOT = next((p for p in [CWD, *CWD.parents] if (p / "django_api" / "manage.py").is_file()), None)
if ROOT is None:
    ROOT = next((p.parent for p in [CWD, *CWD.parents] if (p / "manage.py").is_file() and (p / "ipl_api").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the IPL repository root or the django_api directory.")

DJANGO_DIR = ROOT / "django_api" if (ROOT / "django_api" / "manage.py").is_file() else ROOT
APP_DIR = DJANGO_DIR / "ipl_api"
APP_URLS = APP_DIR / "urls.py"
PROJECT_URLS = DJANGO_DIR / "config" / "urls.py"
DOCS = ROOT / "docs"
DOCS.mkdir(parents=True, exist_ok=True)
BACKUP_DIR = ROOT / "backups" / ("day28_full_" + datetime.datetime.now().strftime("%Y%m%d_%H%M%S"))
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
if not APP_URLS.is_file():
    raise FileNotFoundError(f"App URL file not found: {APP_URLS}")

print("Repository:", ROOT)
print("Django directory:", DJANGO_DIR)
print("App directory:", APP_DIR)
print("Backup directory:", BACKUP_DIR)


Repository: D:\Git\Personal\ipl-player-performance
Django directory: D:\Git\Personal\ipl-player-performance\django_api
App directory: D:\Git\Personal\ipl-player-performance\django_api\ipl_api
Backup directory: D:\Git\Personal\ipl-player-performance\backups\day28_full_20261023_172310


## 1. Back up files and locate the existing dashboard template

In [ ]:
def backup_file(path):
    if path.is_file():
        dest = BACKUP_DIR / path.relative_to(ROOT)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(path, dest)
        print("Backed up:", path.relative_to(ROOT))

backup_file(APP_URLS)
backup_file(PROJECT_URLS)
template_root = APP_DIR / "templates"
templates = list(template_root.rglob("*.html")) if template_root.exists() else []
matches = []
for p in templates:
    try:
        text = p.read_text(encoding="utf-8")
    except Exception:
        continue
    if "Player Performance Dashboard" in text or "Search players" in text or "Refresh summary" in text:
        matches.append(p)
if not matches:
    print("HTML templates found:", [str(p) for p in templates])
    raise FileNotFoundError("Could not identify dashboard template. Update the template selection logic to match your actual template.")
matches.sort(key=lambda p: ("dashboard_day25" not in p.name, len(p.parts)))
dashboard_template = matches[0]
backup_file(dashboard_template)
app_urls_text = APP_URLS.read_text(encoding="utf-8")
print("Dashboard template:", dashboard_template.relative_to(ROOT))
print("\nCurrent app urls.py:\n", app_urls_text)


Backed up: django_api\ipl_api\urls.py
Backed up: django_api\config\urls.py
Backed up: django_api\ipl_api\templates\ipl_api\dashboard_day25.html
Dashboard template: django_api\ipl_api\templates\ipl_api\dashboard_day25.html

Current app urls.py:
 from django.urls import path
from .views import PlayerListAPIView, PlayerDetailAPIView

from .dashboard_views_day26 import dashboard_day25
from .eda_summary_day26 import eda_summary_day26
from .eda_views import EDABattingAPIView, EDABowlingAPIView, EDASeasonAPIView, EDAVenueAPIView, EDAOppositionAPIView, EDARecentFormAPIView
from .prediction_views import BattingPredictionAPIView, BowlingPredictionAPIView, UnifiedPredictionAPIView
from .batch_prediction_views import BatchPredictionAPIView
from .dashboard_views_day28 import dashboard_day28
from .prediction_center_views_day28 import prediction_center_day28
from .player_name_prediction_views_day28 import player_name_prediction_day28, player_name_options_day28, model_metrics_day28
urlpatterns = [
   

## 2. Detect actual prediction and EDA routes

In [ ]:
def route_exists(text, fragment):
    pattern = r"path\(\s*['\"]" + re.escape(fragment) + r"['\"]"
    return bool(re.search(pattern, text))

route_specs = [
    ("batting", "predict/batting/", "/api/predict/batting/", "Batting Prediction", "Estimate next-match runs."),
    ("bowling", "predict/bowling/", "/api/predict/bowling/", "Bowling Prediction", "Estimate wickets and runs conceded."),
    ("batch", "predict/batch/", "/api/predict/batch/", "Batch Prediction", "Submit multiple player records."),
    ("batch_alt", "batch/predict/", "/api/batch/predict/", "Batch Prediction", "Alternate batch route."),
    ("unified", "predict/", "/api/predict/", "Unified Prediction API", "Use the endpoint's documented schema."),
    ("summary", "eda/summary/", "/api/eda/summary/", "EDA Summary", "Historical dataset summary."),
    ("batting_eda", "eda/batting/", "/api/eda/batting/", "Batting EDA", "Batting analytics."),
    ("bowling_eda", "eda/bowling/", "/api/eda/bowling/", "Bowling EDA", "Bowling analytics."),
    ("recent", "eda/recent-form/", "/api/eda/recent-form/", "Recent Form", "Recent player performance."),
]
routes = {}
for key, fragment, url, label, description in route_specs:
    routes[key] = {"fragment": fragment, "url": url, "label": label,
                   "description": description, "registered": route_exists(app_urls_text, fragment)}
    print(label, "=>", "REGISTERED" if routes[key]["registered"] else "NOT FOUND", url)
if routes["batch"]["registered"]:
    routes["batch_alt"]["registered"] = False


Batting Prediction => REGISTERED /api/predict/batting/
Bowling Prediction => REGISTERED /api/predict/bowling/
Batch Prediction => REGISTERED /api/predict/batch/
Batch Prediction => NOT FOUND /api/batch/predict/
Unified Prediction API => REGISTERED /api/predict/
EDA Summary => REGISTERED /api/eda/summary/
Batting EDA => REGISTERED /api/eda/batting/
Bowling EDA => REGISTERED /api/eda/bowling/
Recent Form => REGISTERED /api/eda/recent-form/


## 3. Create the Prediction Center template, view, JavaScript and CSS

In [ ]:
TEMPLATE_DIR = APP_DIR / "templates" / "ipl_api"
JS_DIR = APP_DIR / "static" / "ipl_api" / "js"
CSS_DIR = APP_DIR / "static" / "ipl_api" / "css"
for folder in (TEMPLATE_DIR, JS_DIR, CSS_DIR):
    folder.mkdir(parents=True, exist_ok=True)
PRED_TEMPLATE = TEMPLATE_DIR / "prediction_center_day28.html"
PRED_VIEW = APP_DIR / "prediction_center_views_day28.py"
PRED_JS = JS_DIR / "prediction_center_day28.js"
PRED_CSS = CSS_DIR / "prediction_center_day28.css"
for p in (PRED_TEMPLATE, PRED_VIEW, PRED_JS, PRED_CSS):
    backup_file(p)

PRED_VIEW.write_text(
    "from django.shortcuts import render\n\n"
    "def prediction_center_day28(request):\n"
    "    return render(request, 'ipl_api/prediction_center_day28.html')\n",
    encoding="utf-8"
)


PLAYER_VIEW = APP_DIR / "player_name_prediction_views_day28.py"
backup_file(PLAYER_VIEW)
player_view_code = r'''
import csv
import json
from pathlib import Path

from django.conf import settings
from django.http import JsonResponse
from django.views.decorators.http import require_GET, require_POST

ROOT_DIR = Path(settings.BASE_DIR).parent
BAT_FEATURES = ROOT_DIR / "data" / "processed" / "batting_next_match_features.csv"
BOWL_FEATURES = ROOT_DIR / "data" / "processed" / "bowling_next_match_features.csv"
TARGETS = {
    "next_match_runs": (BAT_FEATURES, "player_name"),
    "next_match_wickets": (BOWL_FEATURES, "player_name"),
    "next_match_runs_conceded": (BOWL_FEATURES, "player_name"),
}

def _read_rows(path):
    if not path.is_file():
        raise FileNotFoundError(f"Feature dataset not found: {path}")
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        return list(csv.DictReader(handle))

def _find_model_config(target):
    from .prediction_views import MODEL_CONFIGS
    return next((item for item in MODEL_CONFIGS if item.get("target") == target), None)

def _as_number(value):
    if value is None or value == "":
        return None
    try:
        number = float(value)
        if number != number or number in (float("inf"), float("-inf")):
            return None
        return number
    except (TypeError, ValueError):
        return None

@require_GET
def player_name_options_day28(request):
    query = request.GET.get("q", "").strip().casefold()
    if not query:
        return JsonResponse({"query": query, "results": []})
    path = BAT_FEATURES
    try:
        rows = _read_rows(path)
    except FileNotFoundError as exc:
        return JsonResponse({"detail": str(exc)}, status=503)
    names = sorted({r.get("player_name", "").strip() for r in rows
                    if query in r.get("player_name", "").strip().casefold()})
    return JsonResponse({"query": query, "results": names[:25]})

@require_POST
def player_name_prediction_day28(request):
    try:
        body = json.loads(request.body.decode("utf-8"))
    except (ValueError, UnicodeDecodeError):
        return JsonResponse({"detail": "Request body must be valid JSON."}, status=400)
    if not isinstance(body, dict):
        return JsonResponse({"detail": "Request body must be a JSON object."}, status=400)
    player_name = str(body.get("player_name", "")).strip()
    target = str(body.get("target", "next_match_runs")).strip()
    if not player_name:
        return JsonResponse({"detail": "player_name is required."}, status=400)
    if target not in TARGETS:
        return JsonResponse({"detail": "Unsupported target.", "allowed_targets": list(TARGETS)}, status=400)
    path, name_column = TARGETS[target]
    try:
        rows = _read_rows(path)
    except FileNotFoundError as exc:
        return JsonResponse({"detail": str(exc)}, status=503)
    matches = [r for r in rows if r.get(name_column, "").strip().casefold() == player_name.casefold()]
    if not matches:
        matches = [r for r in rows if player_name.casefold() in r.get(name_column, "").strip().casefold()]
    if not matches:
        return JsonResponse({"detail": "Player not found in the relevant feature dataset.",
                             "player_name": player_name, "target": target}, status=404)
    # Use the latest available feature row; target/next-match columns are never passed as inputs.
    date_key = "match_date" if any(r.get("match_date") for r in matches) else "date"
    matches.sort(key=lambda r: (r.get(date_key, ""), r.get("match_id", "")))
    row = matches[-1]
    config = _find_model_config(target)
    if config is None:
        return JsonResponse({"detail": f"No saved model configuration for target {target}."}, status=503)
    feature_names = config.get("features", [])
    if not feature_names:
        return JsonResponse({"detail": "Model feature manifest is empty."}, status=503)
    payload = {}
    missing = []
    for feature in feature_names:
        # Never use the known future target or next-match identifiers as model features.
        if feature.startswith("next_match_") or feature in {"match_id", "date", "match_date", "player_name", "player_id"}:
            missing.append(feature)
            continue
        number = _as_number(row.get(feature))
        if number is None:
            missing.append(feature)
        else:
            payload[feature] = number
    if missing:
        return JsonResponse({"detail": "Latest player row cannot supply all required numeric model features.",
                             "missing_or_non_numeric_features": missing,
                             "player_name": row.get("player_name", player_name),
                             "target": target}, status=422)
    try:
        from .prediction_views import _predict
        value, error = _predict(config, payload)
        if error:
            return JsonResponse(error, status=400)
    except Exception as exc:
        return JsonResponse({"detail": "Model inference failed.", "error": str(exc)}, status=500)
    return JsonResponse({
        "player_name": row.get("player_name", player_name),
        "target": target,
        "prediction": value,
        "model": config.get("model_name", config.get("name", "configured model")),
        "feature_row_date": row.get(date_key),
        "features_used": payload,
        "note": "Estimate based on the latest available historical feature row; not a guarantee."
    })

@require_GET
def model_metrics_day28(request):
    # Return only metrics found in actual saved evaluation CSVs; never synthesize metrics.
    reports_root = ROOT_DIR / "reports"
    candidates = []
    for csv_path in reports_root.rglob("*.csv"):
        low = str(csv_path).casefold()
        if any(part in low for part in ("day10", "day12", "day13", "day14")) and "metric" in csv_path.name.casefold() or any(
            token in csv_path.name.casefold() for token in ("model_comparison", "test_metrics", "evaluation_metrics", "metric_summary")
        ):
            try:
                with csv_path.open("r", encoding="utf-8-sig", newline="") as handle:
                    reader = csv.DictReader(handle)
                    rows = list(reader)
                if rows and reader.fieldnames:
                    candidates.append({"file": str(csv_path.relative_to(ROOT_DIR)), "columns": reader.fieldnames, "rows": rows})
            except (OSError, csv.Error):
                continue
    if not candidates:
        return JsonResponse({"detail": "No saved evaluation CSV containing model metrics was found.",
                             "searched": "reports/day10, day12, day13 and day14"}, status=404)
    return JsonResponse({"source": "saved evaluation CSV files", "reports": candidates})
'''
PLAYER_VIEW.write_text(player_view_code, encoding="utf-8")
print("Created:", PLAYER_VIEW.relative_to(ROOT))

html = '''<!doctype html>
<html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>IPL Player Analytics — Prediction Center</title>
<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.3/dist/css/bootstrap.min.css" rel="stylesheet">
<link rel="stylesheet" href="/static/ipl_api/css/prediction_center_day28.css">
</head><body>
<nav class="navbar navbar-dark bg-dark px-3">
<a class="navbar-brand" href="/api/dashboard/">IPL Player Analytics</a>
<span class="text-light small">Prediction Center • EDA • Model Evaluation</span></nav>
<main class="container-fluid py-4">
<div class="d-flex flex-wrap justify-content-between align-items-center gap-2 mb-3">
<div><h1 class="h3 mb-1">Prediction Center</h1><p class="text-secondary mb-0">Explore registered APIs and submit prediction payloads.</p></div>
<a class="btn btn-outline-secondary" href="/api/dashboard/">Back to main dashboard</a></div>
<div class="alert alert-info">Use the exact input feature names and types required by your model manifest and prediction API. This page does not guess missing features.</div>
<section class="card mb-4"><div class="card-header fw-semibold">Registered endpoints</div>
<div class="card-body"><div id="endpointCards" class="row row-cols-1 row-cols-md-2 row-cols-xl-3 g-3"></div></div></section>
<section class="card mb-4"><div class="card-header fw-semibold">Player-name prediction</div><div class="card-body">
<p class="text-secondary">Enter a player name and target. The server builds inputs from the latest historical feature row.</p>
<div class="row g-3"><div class="col-md-6"><label for="playerName" class="form-label">Player name</label><input id="playerName" class="form-control" placeholder="e.g. V Kohli"></div>
<div class="col-md-6"><label for="playerTarget" class="form-label">Prediction target</label><select id="playerTarget" class="form-select">
<option value="next_match_runs">Next-match batting runs</option><option value="next_match_wickets">Next-match wickets</option><option value="next_match_runs_conceded">Next-match runs conceded</option></select></div></div>
<button id="runPlayerPrediction" class="btn btn-primary mt-3">Predict by player name</button><pre id="playerPredictionResult" class="result-panel mt-3">No request submitted.</pre>
</div></section>
<section class="card mb-4"><div class="card-header fw-semibold">JSON single prediction (advanced)</div><div class="card-body">
<label for="singleEndpoint" class="form-label">Prediction endpoint</label><select id="singleEndpoint" class="form-select mb-3"></select>
<label for="singlePayload" class="form-label">Request JSON</label><textarea id="singlePayload" class="form-control code-input mb-3" rows="8" placeholder="Paste the exact valid JSON payload for the selected endpoint."></textarea>
<button id="runSingle" class="btn btn-primary">Run prediction</button><pre id="singleResult" class="result-panel mt-3">No request submitted.</pre>
</div></section>
<section class="card mb-4"><div class="card-header fw-semibold">Batch prediction</div><div class="card-body">
<label for="batchEndpoint" class="form-label">Batch endpoint</label><select id="batchEndpoint" class="form-select mb-3"></select>
<label for="batchPayload" class="form-label">Batch JSON payload</label><textarea id="batchPayload" class="form-control code-input mb-3" rows="8" placeholder="Paste the exact JSON payload accepted by the batch view."></textarea>
<button id="runBatch" class="btn btn-primary">Run batch analysis</button><pre id="batchResult" class="result-panel mt-3">No request submitted.</pre>
</div></section>
<section class="card"><div class="card-header fw-semibold">Model metrics</div><div class="card-body">
<p class="text-secondary">MAE, RMSE and R² must come from a real metrics endpoint or saved evaluation reports; values are not fabricated.</p>
<label for="metricsUrl" class="form-label">Metrics endpoint, if registered</label><input id="metricsUrl" class="form-control mb-3" value="/api/eda/model-metrics/">
<button id="loadMetrics" class="btn btn-outline-primary">Load metrics</button><pre id="metricsResult" class="result-panel mt-3">Metrics not loaded.</pre>
</div></section>
<p class="small text-secondary mt-4">Predictions are estimates, not guarantees.</p></main>
<script src="/static/ipl_api/js/prediction_center_day28.js"></script></body></html>'''
PRED_TEMPLATE.write_text(html, encoding="utf-8")
print("Created:", PRED_TEMPLATE.relative_to(ROOT))
print("Created:", PRED_VIEW.relative_to(ROOT))


Backed up: django_api\ipl_api\templates\ipl_api\prediction_center_day28.html
Backed up: django_api\ipl_api\prediction_center_views_day28.py
Backed up: django_api\ipl_api\static\ipl_api\js\prediction_center_day28.js
Backed up: django_api\ipl_api\static\ipl_api\css\prediction_center_day28.css
Backed up: django_api\ipl_api\player_name_prediction_views_day28.py
Created: django_api\ipl_api\player_name_prediction_views_day28.py
Created: django_api\ipl_api\templates\ipl_api\prediction_center_day28.html
Created: django_api\ipl_api\prediction_center_views_day28.py


## 4. Wire the frontend to the route inventory

In [ ]:
route_json = json.dumps(routes, ensure_ascii=False)
js = r'''(() => {
const ROUTES = __ROUTES__;
const $ = id => document.getElementById(id);
const show = (id, v) => { $(id).textContent = typeof v === "string" ? v : JSON.stringify(v, null, 2); };
Object.values(ROUTES).forEach(route => {
  const col = document.createElement("div"); col.className = "col";
  const card = document.createElement("div"); card.className = "endpoint-card h-100";
  const title = document.createElement("strong"); title.textContent = route.label;
  const desc = document.createElement("p"); desc.className = "small text-secondary mt-2"; desc.textContent = route.description;
  const status = document.createElement("span"); status.className = route.registered ? "badge text-bg-success" : "badge text-bg-secondary";
  status.textContent = route.registered ? "Registered" : "Not registered";
  card.append(title, desc, status);
  if (route.registered) { const a = document.createElement("a"); a.href = route.url; a.className = "d-block mt-2"; a.textContent = "Open API endpoint"; card.append(a); }
  col.append(card); $("endpointCards").append(col);
});
function fill(id, keys, emptyLabel) {
  const select = $(id); select.innerHTML = "";
  const first = document.createElement("option"); first.value = ""; first.textContent = emptyLabel; select.append(first);
  keys.filter(k => ROUTES[k] && ROUTES[k].registered).forEach(k => {
    const o = document.createElement("option"); o.value = ROUTES[k].url; o.textContent = ROUTES[k].label + " — " + ROUTES[k].url; select.append(o);
  });
  if (select.options.length === 1) select.disabled = true;
}
fill("singleEndpoint", ["batting","bowling","unified"], "No registered prediction endpoint found");
fill("batchEndpoint", ["batch","batch_alt"], "No registered batch endpoint found");
async function req(url, options={}) {
  const r = await fetch(url, {headers: {"Accept":"application/json", ...(options.headers||{})}, ...options});
  const t = await r.text(); let data;
  try { data = t ? JSON.parse(t) : {}; } catch { throw new Error("HTTP " + r.status + ": response was not JSON: " + t.slice(0,250)); }
  if (!r.ok) throw new Error("HTTP " + r.status + ": " + JSON.stringify(data));
  return data;
}
async function post(endpointId, payloadId, resultId) {
  const url = $(endpointId).value;
  if (!url) { show(resultId, "No registered endpoint selected. Inspect ipl_api/urls.py."); return; }
  let payload; try { payload = JSON.parse($(payloadId).value); } catch(e) { show(resultId, "Invalid JSON: " + e.message); return; }
  show(resultId, "Requesting…");
  try { show(resultId, await req(url, {method:"POST", headers:{"Content-Type":"application/json"}, body:JSON.stringify(payload)})); }
  catch(e) { show(resultId, e.message); }
}
$("runSingle").addEventListener("click", () => post("singleEndpoint","singlePayload","singleResult"));
$("runPlayerPrediction").addEventListener("click", async () => {
  const player_name = $("playerName").value.trim();
  const target = $("playerTarget").value;
  if (!player_name) { show("playerPredictionResult", "Enter a player name first."); return; }
  show("playerPredictionResult", "Requesting prediction…");
  try {
    const result = await req("/api/predict/player/", {
      method: "POST",
      headers: {"Content-Type":"application/json"},
      body: JSON.stringify({player_name, target})
    });
    show("playerPredictionResult", result);
  } catch (e) { show("playerPredictionResult", e.message); }
});

$("runBatch").addEventListener("click", () => post("batchEndpoint","batchPayload","batchResult"));
$("loadMetrics").addEventListener("click", async () => {
  show("metricsResult","Loading…");
  try { show("metricsResult", await req($("metricsUrl").value.trim())); }
  catch(e) { show("metricsResult", e.message + "\nIf this route is not registered, compare reports/day10, day12, day13 and day14."); }
});
})();'''.replace("__ROUTES__", route_json)
PRED_JS.write_text(js, encoding="utf-8")

css = '''body { background: #f4f6fb; color: #172033; }
.card { border: 0; box-shadow: 0 3px 14px rgba(20,35,60,.07); }
.card-header { background: #fff; }
.endpoint-card { background: #fff; border: 1px solid #e4e8f0; border-radius: .65rem; padding: 1rem; }
.code-input, .result-panel { font-family: Consolas, "Courier New", monospace; font-size: .9rem; }
.result-panel { min-height: 72px; max-height: 360px; overflow: auto; white-space: pre-wrap; overflow-wrap: anywhere; background: #111827; color: #e5e7eb; padding: 1rem; border-radius: .4rem; }
'''
PRED_CSS.write_text(css, encoding="utf-8")
print("Created:", PRED_JS.relative_to(ROOT))
print("Created:", PRED_CSS.relative_to(ROOT))


Created: django_api\ipl_api\static\ipl_api\js\prediction_center_day28.js
Created: django_api\ipl_api\static\ipl_api\css\prediction_center_day28.css


## 5. Automatically register `/api/predictions/` and add a dashboard button

In [ ]:
app_text = APP_URLS.read_text(encoding="utf-8")
import_line = "from .prediction_center_views_day28 import prediction_center_day28"
route_line = 'path("predictions/", prediction_center_day28, name="ipl-predictions-day28"),'
if import_line not in app_text:
    lines = app_text.splitlines()
    import_idxs = [i for i, line in enumerate(lines) if re.match(r"^\s*(from\s+\S+\s+import\s+|import\s+)", line)]
    insert_at = max(import_idxs) + 1 if import_idxs else 0
    lines.insert(insert_at, import_line)
    app_text = "\n".join(lines) + "\n"
if route_line not in app_text:
    match = re.search(r"urlpatterns\s*=\s*\[", app_text)
    if not match:
        raise ValueError("Could not find urlpatterns = [ in app urls.py; stopped without adding the route.")
    pos = match.end()
    app_text = app_text[:pos] + "\n    " + route_line + app_text[pos:]
APP_URLS.write_text(app_text, encoding="utf-8")
print("Registered route:", route_line)

dashboard_text = dashboard_template.read_text(encoding="utf-8")
button = '<a class="btn btn-outline-primary ms-2" href="/api/predictions/">Prediction Center</a>'
if 'href="/api/predictions/"' not in dashboard_text:
    if re.search(r"</nav>", dashboard_text, re.I):
        dashboard_text = re.sub(r"</nav>", button + "\n</nav>", dashboard_text, count=1, flags=re.I)
    elif re.search(r"</main>", dashboard_text, re.I):
        dashboard_text = re.sub(r"</main>", '<div class="container my-3">' + button + '</div>\n</main>', dashboard_text, count=1, flags=re.I)
    elif re.search(r"</body>", dashboard_text, re.I):
        dashboard_text = re.sub(r"</body>", '<div class="container my-3">' + button + '</div>\n</body>', dashboard_text, count=1, flags=re.I)
    else:
        dashboard_text += "\n" + button
    dashboard_template.write_text(dashboard_text, encoding="utf-8")
    print("Added Prediction Center button to", dashboard_template.relative_to(ROOT))
else:
    print("Prediction Center button already exists.")


# Day 28: register player-name prediction and saved-report metrics routes.
app_text = APP_URLS.read_text(encoding="utf-8")
extra_imports = [
    "from .player_name_prediction_views_day28 import player_name_prediction_day28, player_name_options_day28, model_metrics_day28"
]
for line in extra_imports:
    if line not in app_text:
        lines = app_text.splitlines()
        import_idxs = [i for i, item in enumerate(lines) if re.match(r"^\s*(from\s+\S+\s+import\s+|import\s+)", item)]
        lines.insert(max(import_idxs) + 1 if import_idxs else 0, line)
        app_text = "\n".join(lines) + "\n"
extra_routes = [
    ('path("predict/player/", player_name_prediction_day28, name="predict-player-name-day28"),', 'predict/player/'),
    ('path("players/search-options/", player_name_options_day28, name="player-name-options-day28"),', 'players/search-options/'),
    ('path("eda/model-metrics/", model_metrics_day28, name="model-metrics-day28"),', 'eda/model-metrics/'),
]
for route, fragment in extra_routes:
    if not route_exists(app_text, fragment):
        match = re.search(r"urlpatterns\s*=\s*\[", app_text)
        if not match:
            raise ValueError("Could not find urlpatterns = [ while registering Day 28 routes.")
        pos = match.end()
        app_text = app_text[:pos] + "\n    " + route + app_text[pos:]
APP_URLS.write_text(app_text, encoding="utf-8")
print("Registered player-name prediction, name search and metrics routes.")


Registered route: path("predictions/", prediction_center_day28, name="ipl-predictions-day28"),
Prediction Center button already exists.
Registered player-name prediction, name search and metrics routes.


## 6. Run Django system check and smoke tests

In [ ]:
check = subprocess.run([sys.executable, "manage.py", "check"], cwd=DJANGO_DIR, capture_output=True, text=True)
print("manage.py check exit code:", check.returncode)
print(check.stdout)
if check.stderr:
    print(check.stderr)
if check.returncode != 0:
    raise RuntimeError("Django system check failed. Review the output before continuing.")


manage.py check exit code: 0
System check identified no issues (0 silenced).



In [ ]:
smoke_code = r'''
import os
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings")
import django
django.setup()
from django.test import Client
c = Client()
for path in ["/api/dashboard/", "/api/predictions/", "/api/eda/summary/", "/api/eda/model-metrics/", "/api/players/search-options/?q=vir"]:
    r = c.get(path, HTTP_ACCEPT="application/json")
    print(path, "=>", r.status_code, r.get("Content-Type"))
'''
smoke = subprocess.run([sys.executable, "-c", smoke_code], cwd=DJANGO_DIR, capture_output=True, text=True)
print("Smoke-test exit code:", smoke.returncode)
print(smoke.stdout)
if smoke.stderr:
    print(smoke.stderr[-3000:])


Smoke-test exit code: 0
/api/dashboard/ => 400 text/plain; charset=utf-8
/api/predictions/ => 400 text/plain; charset=utf-8
/api/eda/summary/ => 400 text/plain; charset=utf-8
/api/eda/model-metrics/ => 400 text/plain; charset=utf-8
/api/players/search-options/?q=vir => 400 text/plain; charset=utf-8

nt call last):
  File "c:\Users\Vinod\anaconda3\envs\ipl-player-performance\Lib\site-packages\django\core\handlers\exception.py", line 55, in inner
    response = get_response(request)
               ^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vinod\anaconda3\envs\ipl-player-performance\Lib\site-packages\django\utils\deprecation.py", line 309, in __call__
    response = self.process_request(request)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vinod\anaconda3\envs\ipl-player-performance\Lib\site-packages\django\middleware\common.py", line 48, in process_request
    host = request.get_host()
           ^^^^^^^^^^^^^^^^^^
  File "c:\Users\Vinod\anaconda3\envs\ipl-player-performanc

## 7. Generate the Day 28 report

In [ ]:
report_path = DOCS / "day28_working_report.md"
route_summary = "\n".join(
    f"- {item['label']}: {'registered' if item['registered'] else 'not found'} — `{item['url']}`"
    for item in routes.values()
)
report = f'''# Day 28 — Prediction Center Report

Generated: {datetime.datetime.now().isoformat(timespec="seconds")}

## Files created or changed
- `{PRED_TEMPLATE.relative_to(ROOT)}`
- `{PRED_VIEW.relative_to(ROOT)}`
- `{PRED_JS.relative_to(ROOT)}`
- `{PRED_CSS.relative_to(ROOT)}`
- `{dashboard_template.relative_to(ROOT)}`
- `{APP_URLS.relative_to(ROOT)}`

## Django system check
Exit code: {check.returncode}

{check.stdout.strip()}

## Smoke test
Exit code: {smoke.returncode}

{smoke.stdout.strip()}

## Routes detected before adding the Prediction Center
{route_summary}

## Important
A registered URL is not proof that model inference succeeds. Test each prediction endpoint with a valid JSON payload from the actual view/model manifest. Metrics must come from saved evaluation reports or a real metrics endpoint.

Backup folder: `{BACKUP_DIR.relative_to(ROOT)}`
'''
report_path.write_text(report, encoding="utf-8")
print("Report saved:", report_path)
print(report)


Report saved: D:\Git\Personal\ipl-player-performance\docs\day28_working_report.md
# Day 28 — Prediction Center Report

Generated: 2026-10-23T17:23:15

## Files created or changed
- `django_api\ipl_api\templates\ipl_api\prediction_center_day28.html`
- `django_api\ipl_api\prediction_center_views_day28.py`
- `django_api\ipl_api\static\ipl_api\js\prediction_center_day28.js`
- `django_api\ipl_api\static\ipl_api\css\prediction_center_day28.css`
- `django_api\ipl_api\templates\ipl_api\dashboard_day25.html`
- `django_api\ipl_api\urls.py`

## Django system check
Exit code: 0

System check identified no issues (0 silenced).

## Smoke test
Exit code: 0

/api/dashboard/ => 400 text/plain; charset=utf-8
/api/predictions/ => 400 text/plain; charset=utf-8
/api/eda/summary/ => 400 text/plain; charset=utf-8
/api/eda/model-metrics/ => 400 text/plain; charset=utf-8
/api/players/search-options/?q=vir => 400 text/plain; charset=utf-8

## Routes detected before adding the Prediction Center
- Batting Predict

## Day 28 acceptance checklist

- [ ] All cells run without errors.
- [ ] `python manage.py check` passes.
- [ ] Existing dashboard still loads at `/api/dashboard/`.
- [ ] Prediction Center loads at `/api/predictions/`.
- [ ] Summary endpoint still returns JSON.
- [ ] Single prediction tested with valid payload.
- [ ] Batch prediction tested with valid multiple-record payload.
- [ ] MAE/RMSE/R² checked against saved reports.
- [ ] `docs/day28_working_report.md` reviewed.

Do not mark prediction inference or model comparison complete until those endpoints have been tested with actual requests.


## 8. Separate ML and Deep Learning prediction panels

This cell labels the existing player-name workflow as ML and adds a distinct DL panel. It inspects `ipl_api/urls.py` for a dedicated DL route. If no such route exists, the DL panel stays disabled and clearly reports that; it never sends a DL request to the ML endpoint.

In [ ]:

from pathlib import Path
import re, json

template_path = APP_DIR / "templates" / "ipl_api" / "prediction_center_day28.html"
js_path = APP_DIR / "static" / "ipl_api" / "js" / "prediction_center_day28.js"
urls_text = APP_URLS.read_text(encoding="utf-8")
html_text = template_path.read_text(encoding="utf-8")
js_text = js_path.read_text(encoding="utf-8")

html_text = html_text.replace(
    '<div class="card-header fw-semibold">Player-name prediction</div>',
    '<div class="card-header fw-semibold">Machine Learning (ML) — player-name prediction</div>'
).replace("Predict by player name", "Run ML prediction")

dl_route = None
for fragment in ("predict/deep-learning/player/", "predict/dl/player/", "predict/deep-learning/", "predict/dl/"):
    if re.search(r'path\(\s*["\\\']' + re.escape(fragment), urls_text):
        dl_route = "/api/" + fragment
        break
dl_registered = bool(dl_route)

dl_panel = f"""
<section class="card mb-4" id="dlPredictionCard">
  <div class="card-header fw-semibold">Deep Learning (DL) prediction</div>
  <div class="card-body">
    <p class="text-secondary">This panel is separate from the ML pipeline and requires the saved sequence model plus its matching preprocessing artifacts.</p>
    <div class="alert {"alert-success" if dl_registered else "alert-warning"}" id="dlStatus">
      {"DL endpoint detected: " + dl_route if dl_registered else "No dedicated DL prediction route was found in ipl_api/urls.py. The panel is disabled rather than returning an ML result as if it were DL."}
    </div>
    <div class="row g-3">
      <div class="col-md-6">
        <label for="dlPlayerName" class="form-label">Player name (if supported by the DL endpoint)</label>
        <input id="dlPlayerName" class="form-control" placeholder="e.g. V Kohli" {"disabled" if not dl_registered else ""}>
      </div>
      <div class="col-md-6">
        <label for="dlTarget" class="form-label">Prediction target</label>
        <select id="dlTarget" class="form-select" {"disabled" if not dl_registered else ""}>
          <option value="next_match_runs">Next-match batting runs</option>
          <option value="next_match_wickets">Next-match wickets</option>
          <option value="next_match_runs_conceded">Next-match runs conceded</option>
        </select>
      </div>
    </div>
    <button id="runDlPlayerPrediction" class="btn btn-primary mt-3" {"disabled" if not dl_registered else ""}>Run DL prediction</button>
    <pre id="dlPredictionResult" class="result-panel mt-3">{"No DL request submitted." if dl_registered else "DL endpoint is not registered."}</pre>
    <p class="small text-secondary mb-0">A real DL endpoint must build the expected chronological sequence and apply the matching imputer/scaler. It must not reuse ML inputs unless the saved DL manifest explicitly requires them.</p>
  </div>
</section>
"""
if 'id="dlPredictionCard"' not in html_text:
    marker = '<section class="card mb-4"><div class="card-header fw-semibold">JSON single prediction (advanced)</div>'
    if marker in html_text:
        html_text = html_text.replace(marker, dl_panel + "\n" + marker, 1)
    else:
        html_text = html_text.replace("</main>", dl_panel + "\n</main>", 1)
template_path.write_text(html_text, encoding="utf-8")

if dl_registered and "runDlPlayerPrediction" not in js_text:
    handler = f"""
const DL_PLAYER_ENDPOINT = {json.dumps(dl_route)};
$("runDlPlayerPrediction").addEventListener("click", async () => {{
  const player_name = $("dlPlayerName").value.trim();
  const target = $("dlTarget").value;
  if (!player_name) {{ show("dlPredictionResult", "Enter a player name first."); return; }}
  show("dlPredictionResult", "Requesting DL prediction…");
  try {{
    show("dlPredictionResult", await req(DL_PLAYER_ENDPOINT, {{
      method: "POST",
      headers: {{"Content-Type":"application/json"}},
      body: JSON.stringify({{player_name, target}})
    }}));
  }} catch (e) {{ show("dlPredictionResult", e.message); }}
}});
"""
    js_text = js_text.replace("})();", handler + "\n})();")
    js_path.write_text(js_text, encoding="utf-8")

print("Updated:", template_path.relative_to(ROOT))
print("ML player-name panel labeled separately.")
print("Dedicated DL endpoint registered:", dl_registered)
print("Detected DL route:", dl_route or "none")


Updated: django_api\ipl_api\templates\ipl_api\prediction_center_day28.html
ML player-name panel labeled separately.
Dedicated DL endpoint registered: False
Detected DL route: none


## 9. Run Django check and write the ML/DL separation report

In [ ]:

check = subprocess.run([sys.executable, "manage.py", "check"], cwd=DJANGO_DIR, capture_output=True, text=True)
print("python manage.py check exit code:", check.returncode)
print(check.stdout)
if check.stderr:
    print(check.stderr)
report = DOCS / "day28_ml_dl_separation_report.md"
report.write_text(
    "# Day 28 — ML and DL Prediction Separation\n\n"
    f"- ML player-name endpoint: `/api/predict/player/` (existing workflow)\n"
    f"- Dedicated DL endpoint registered: `{dl_registered}`\n"
    f"- DL route detected: `{dl_route or 'none'}`\n"
    "- The UI does not claim DL inference is available unless a dedicated route exists.\n"
    "- Confirm sequence length, feature order, target mapping, model path, and matching preprocessing artifacts before enabling DL inference.\n"
    f"- Django check exit code: `{check.returncode}`\n",
    encoding="utf-8"
)
print("Report saved:", report.relative_to(ROOT))


python manage.py check exit code: 0
System check identified no issues (0 silenced).

Report saved: docs\day28_ml_dl_separation_report.md
